# DataFrame desde PostgreSQL
En esta sección se crea el DataFrame principal a partir de la tabla almacenada en PostgreSQL y se realiza una exploración inicial de los datos.

In [9]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv


## Carga de variables de entorno

Se cargan las credenciales de conexión almacenadas en el archivo ".env`" para evitar escribir directamente datos sensibles en el notebook.

In [10]:
load_dotenv()
DB_USER = os.getenv("POSTGRES_USER")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")
DB_NAME = os.getenv("POSTGRES_DB")
DB_PORT = os.getenv("POSTGRES_PORT")
DB_HOST = "127.0.0.1"


## Creación del motor de conexión

Se construye el engine de SQLAlchemy para establecer la conexión con PostgreSQL.

In [11]:
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)


## Validación de la conexión

Se ejecuta una consulta simple para comprobar que la conexión a la base de datos funciona correctamente.

In [12]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print("Conexión exitosa:", result.scalar())

Conexión exitosa: 1



## Carga de la tabla como DataFrame

Se consulta la tabla `sale_report` del esquema `public` y se almacena el resultado en un DataFrame para su posterior análisis.


In [13]:
df_bd = pd.read_sql("SELECT * FROM public.sale_report", engine)

## Primeras 5 filas del DataFrame
Se visualizan las primeras cinco filas del DataFrame para conocer la estructura inicial de la información.


In [14]:
df_bd.head()

,index,SKU Code,Design No.,stock,category,size,color
0,0,AN201-RED-L,AN201,5.0,AN : LEGGINGS,L,Red
1,1,AN201-RED-M,AN201,5.0,AN : LEGGINGS,M,Red
2,2,AN201-RED-S,AN201,3.0,AN : LEGGINGS,S,Red
3,3,AN201-RED-XL,AN201,6.0,AN : LEGGINGS,XL,Red
4,4,AN201-RED-XXL,AN201,3.0,AN : LEGGINGS,XXL,Red


## Columnas con valores nulos
Se identifican las columnas que presentan al menos un valor nulo

In [15]:
columnas_con_nulos = df_bd.columns[df_bd.isnull().any()]
print("Columnas con valores nulos:")
print(columnas_con_nulos.tolist())

Columnas con valores nulos:
['SKU Code', 'Design No.', 'stock', 'category', 'size', 'color']



## Conteo de valores faltantes por columna
Se cuantifica cuántos valores faltan en cada columna del DataFrame.

In [16]:
faltantes_por_columna = df_bd.isnull().sum()
print("Valores faltantes por columna:")
print(faltantes_por_columna)

Valores faltantes por columna:
index          0
SKU Code      83
Design No.    36
stock         36
category      45
size          36
color         45
dtype: int64



## Tipos de datos y selección de variables numéricas
Se identifican las columnas numéricas disponibles para calcular estadísticas descriptivas.

In [17]:
print(df_bd.dtypes)
print(df_bd.columns.tolist())

index           int64
SKU Code          str
Design No.        str
stock         float64
category          str
size              str
color             str
dtype: object
['index', 'SKU Code', 'Design No.', 'stock', 'category', 'size', 'color']


## Estadísticas de la variable numérica seleccionada
Se calcula el promedio, el valor máximo y el valor mínimo de la variable `stock`.

In [18]:
df_bd["stock"].agg(["mean", "max", "min"])

mean      26.246454
max     1234.000000
min        0.000000
Name: stock, dtype: float64


## Máximo y mínimo agrupado
Se calcula el valor máximo y mínimo de la variable `stock` agrupando

In [19]:
df_bd.groupby("category")["stock"].agg(["max", "min"])

,max,min
category,,
AN : LEGGINGS,23.0,0.0
BLOUSE,628.0,0.0
BOTTOM,4.0,0.0
CARDIGAN,11.0,0.0
CROP TOP,45.0,2.0
CROP TOP WITH PLAZZO,232.0,11.0
DRESS,441.0,0.0
JUMPSUIT,11.0,2.0
KURTA,1234.0,0.0
